
# SVG Generation from Text Prompts

This notebook is designed for your competition constraints:

- public pretrained model
- model size **<= 2B**
- **LoRA / QLoRA** fine-tuning
- **no external dataset**
- output `submission.csv` with columns `id, svg`

### Main choices
- Base model: **Qwen/Qwen2.5-1.5B-Instruct**
- 4-bit QLoRA
- moderate SVG cleaning
- standard `transformers.Trainer` instead of `trl.SFTTrainer` to avoid the version issues you hit
- greedy batched inference
- two-pass recovery only for failed rows


In [ ]:

# ============================================================
# CELL 0 - INSTALL
# ============================================================
!pip -q install -U transformers peft accelerate bitsandbytes datasets scikit-learn lxml cairosvg


In [ ]:

# ============================================================
# CELL 1 - CONFIG
# ============================================================

# ----- file names -----
TRAIN_CSV = "train.csv"
TEST_CSV = "test.csv"
SAMPLE_SUB = "sample_submission.csv"

# ----- model -----
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# ----- data subset -----
NUM_TRAIN_SAMPLES = 12000     # good L4 starting point
VAL_FRAC = 0.05
SEED = 42

# ----- svg filtering -----
MIN_SVG_CHARS = 60
MAX_TRAIN_SVG_CHARS = 5200
MAX_SUBMIT_SVG_CHARS = 8000
MAX_PATH_ELEMENTS = 256

# ----- training -----
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 2
GRAD_ACCUM = 8
NUM_EPOCHS = 2
LR = 1e-4
WARMUP_STEPS = 80
SAVE_STEPS = 250
EVAL_STEPS = 250
OUTPUT_DIR = "./svg_qwen15b_instruct_ckpt"
ADAPTER_DIR = "./svg_qwen15b_instruct_adapter"

# ----- inference -----
GEN_MAX_NEW_TOKENS = 650
GEN_MAX_NEW_TOKENS_RETRY = 900
INFERENCE_BATCH = 8
INFERENCE_BATCH_RETRY = 4

# ----- output -----
SUBMISSION_PATH = "submission.csv"

print("Config ready.")
print("MODEL_ID =", MODEL_ID)
print("NUM_TRAIN_SAMPLES =", NUM_TRAIN_SAMPLES)


In [ ]:

# ============================================================
# CELL 2 - IMPORTS / SEED / DEVICE
# ============================================================

import os
import re
import gc
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import xml.etree.ElementTree as ET

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader


In [ ]:

# ============================================================
# CELL 3 - LOAD DATA
# ============================================================

# train_df = pd.read_csv(TRAIN_CSV, engine='python')
# test_df = pd.read_csv(TEST_CSV, engine='python')

# print("train shape:", train_df.shape)
# print("test shape :", test_df.shape)
# print("train cols :", train_df.columns.tolist())
# print("test cols  :", test_df.columns.tolist())

# assert {"prompt", "svg"}.issubset(train_df.columns), "train.csv must contain prompt and svg"
# assert {"id", "prompt"}.issubset(test_df.columns), "test.csv must contain id and prompt"

# display(train_df.head(2))
# display(test_df.head(2))

train_df = pd.read_csv(
    TRAIN_CSV,
    engine='python',
    on_bad_lines='skip',
    quoting=0,              # QUOTE_MINIMAL
    encoding='utf-8',
    encoding_errors='replace',
)
test_df = pd.read_csv(
    TEST_CSV,
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8',
    encoding_errors='replace',
)

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
print("train cols :", train_df.columns.tolist())
print("test cols  :", test_df.columns.tolist())

assert {"prompt", "svg"}.issubset(train_df.columns), "train.csv must contain prompt and svg"
assert {"id", "prompt"}.issubset(test_df.columns), "test.csv must contain id and prompt"

display(train_df.head(2))
display(test_df.head(2))



In [ ]:

# ============================================================
# CELL 4 - SVG HELPERS
# ============================================================

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clipPath",
    "mask", "linearGradient", "radialGradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256" '
    'width="256" height="256"><rect width="256" height="256" fill="white"/></svg>'
)

def strip_ns(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def count_path_elements(svg):
    return len(re.findall(r"<path(?=[\\s>/])", svg or "", flags=re.IGNORECASE))

def extract_svg_block(text):
    if not isinstance(text, str):
        return None
    start = text.find("<svg")
    if start == -1:
        return None
    end = text.rfind("</svg>")
    if end == -1:
        return text[start:] + "</svg>"
    return text[start:end + len("</svg>")]

def compact_ws(text):
    text = re.sub(r">\\s+<", "><", text)
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

def quick_train_clean(svg):
    """
    Moderate cleaning:
    - keep training signal
    - reject obviously broken or huge targets
    - do not over-rewrite geometry
    """
    if not isinstance(svg, str):
        return None
    svg = extract_svg_block(svg.strip())
    if svg is None:
        return None
    svg = re.sub(r"<!--.*?-->", "", svg, flags=re.DOTALL)
    svg = compact_ws(svg)

    if not svg.startswith("<svg"):
        return None
    if len(svg) < MIN_SVG_CHARS:
        return None
    if len(svg) > MAX_TRAIN_SVG_CHARS:
        return None
    if count_path_elements(svg) > MAX_PATH_ELEMENTS:
        return None

    # Keep moderate strictness: parse if possible; reject only if XML is clearly broken.
    try:
        root = ET.fromstring(svg)
        if strip_ns(root.tag) != "svg":
            return None
    except Exception:
        return None

    return svg

def validate_submit_svg(svg):
    if not isinstance(svg, str):
        return False, "not_string"
    if not svg.startswith("<svg"):
        return False, "missing_svg"
    if len(svg) > MAX_SUBMIT_SVG_CHARS:
        return False, "too_long"
    if count_path_elements(svg) > MAX_PATH_ELEMENTS:
        return False, "too_many_paths"
    try:
        root = ET.fromstring(svg)
    except Exception:
        return False, "xml_error"
    if strip_ns(root.tag) != "svg":
        return False, "bad_root"
    for elem in root.iter():
        if strip_ns(elem.tag) not in ALLOWED_TAGS:
            return False, "bad_tag"
    return True, "ok"

def repair_for_submit(svg):
    """
    Keep this lighter than the over-strict version that collapsed your score.
    """
    svg = extract_svg_block(svg)
    if svg is None:
        return FALLBACK_SVG

    svg = compact_ws(svg)

    if "xmlns=" not in svg:
        svg = svg.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if "viewBox=" not in svg:
        svg = re.sub(r"<svg([^>]*)>", r'<svg\\1 viewBox="0 0 256 256">', svg, count=1)

    if len(svg) > MAX_SUBMIT_SVG_CHARS:
        cut = svg[:MAX_SUBMIT_SVG_CHARS - 20].rfind("<")
        if cut > 0:
            svg = svg[:cut] + "</svg>"
        else:
            return FALLBACK_SVG

    if count_path_elements(svg) > MAX_PATH_ELEMENTS:
        return FALLBACK_SVG

    ok, _ = validate_submit_svg(svg)
    return svg if ok else FALLBACK_SVG

print("SVG helper functions ready.")


In [ ]:

# ============================================================
# CELL 5 - CLEAN + SAMPLE TRAIN DATA
# ============================================================

work_df = train_df.copy()
work_df["svg_clean"] = work_df["svg"].apply(quick_train_clean)
before = len(work_df)
work_df = work_df.dropna(subset=["svg_clean"]).reset_index(drop=True)
after = len(work_df)

work_df["svg_len"] = work_df["svg_clean"].str.len()
work_df["path_count"] = work_df["svg_clean"].apply(count_path_elements)
work_df["prompt_len"] = work_df["prompt"].astype(str).str.len()

print(f"Kept {after}/{before} rows after cleaning")
display(work_df[["svg_len", "path_count", "prompt_len"]].describe())

# length-diverse sampling helps more than plain random early rows
if len(work_df) > NUM_TRAIN_SAMPLES:
    q = min(12, work_df["svg_len"].nunique())
    work_df["svg_bin"] = pd.qcut(work_df["svg_len"], q=q, labels=False, duplicates="drop")
    chosen = []
    per_bin = max(1, NUM_TRAIN_SAMPLES // max(1, work_df["svg_bin"].nunique()))
    for b in sorted(work_df["svg_bin"].dropna().unique()):
        part = work_df[work_df["svg_bin"] == b]
        chosen.append(part.sample(n=min(len(part), per_bin), random_state=SEED))
    chosen_df = pd.concat(chosen).drop_duplicates()
    if len(chosen_df) < NUM_TRAIN_SAMPLES:
        need = NUM_TRAIN_SAMPLES - len(chosen_df)
        remain = work_df.drop(chosen_df.index)
        chosen_df = pd.concat([
            chosen_df,
            remain.sample(n=min(need, len(remain)), random_state=SEED)
        ])
    work_df = chosen_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    work_df = work_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print("Final sampled rows:", len(work_df))
display(work_df[["prompt", "svg_clean"]].head(2))


In [ ]:

# ============================================================
# CELL 6 - TOKENIZER + TRAIN TEXT FORMAT
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = (
    "Generate only valid compact SVG. "
    "Output only SVG XML. "
    "Start with <svg and end with </svg>."
)

def format_train_text(prompt, svg):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(prompt)},
        {"role": "assistant", "content": str(svg)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def build_infer_text(prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(prompt)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

example_text = format_train_text(work_df.iloc[0]["prompt"], work_df.iloc[0]["svg_clean"])
print(example_text[:900])


In [ ]:

# ============================================================
# CELL 7 - BUILD HF DATASETS
# ============================================================

train_part, val_part = train_test_split(work_df, test_size=VAL_FRAC, random_state=SEED)

train_ds = Dataset.from_pandas(train_part[["prompt", "svg_clean"]].rename(columns={"svg_clean": "svg"}))
val_ds = Dataset.from_pandas(val_part[["prompt", "svg_clean"]].rename(columns={"svg_clean": "svg"}))

def tokenize_train(example):
    text = format_train_text(example["prompt"], example["svg"])
    tok = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length", # Changed from False to "max_length"
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

train_tok = train_ds.map(tokenize_train, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_train, remove_columns=val_ds.column_names)

print("train rows:", len(train_tok))
print("val rows  :", len(val_tok))
print(train_tok[0].keys())
print("example token length:", len(train_tok[0]["input_ids"]))


In [ ]:

# ============================================================
# CELL 8 - LOAD MODEL + QLoRA
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # safer on L4/Colab than bf16 here
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:

# ============================================================
# CELL 9 - TRAIN
# ============================================================

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

trainer.train()

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to:", ADAPTER_DIR)


In [ ]:

# ============================================================
# CELL 10 - INFERENCE HELPERS
# ============================================================

model.config.use_cache = True
model.eval()

@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=GEN_MAX_NEW_TOKENS, batch_size=None):
    formatted = [build_infer_text(p) for p in prompts]
    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=192,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_lengths = inputs["attention_mask"].sum(dim=1).tolist()
    svgs = []
    for i, out in enumerate(outputs):
        gen_tokens = out[int(input_lengths[i]):]
        raw = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        svg = repair_for_submit(raw)
        svgs.append(svg)
    return svgs

# quick sanity check
sample_prompts = val_part["prompt"].head(5).tolist()
sample_svgs = generate_batch(sample_prompts, max_new_tokens=GEN_MAX_NEW_TOKENS)
for i, (p, s) in enumerate(zip(sample_prompts, sample_svgs), 1):
    ok, reason = validate_submit_svg(s)
    print("="*80)
    print("sample", i, "| valid:", ok, reason, "| len:", len(s), "| paths:", count_path_elements(s))
    print("prompt:", p[:180])
    print("svg:", s[:350])


In [ ]:
# ============================================================
# CELL 11 - INFERENCE (SAFE XML VERSION)
# ============================================================

import re
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch

# IMPORTANT for decoder-only batched generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" '
    'viewBox="0 0 256 256" width="256" height="256">'
    '<rect width="256" height="256" fill="white"/></svg>'
)

def build_prompt_text(prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(prompt)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def extract_svg(raw: str) -> str:
    if not isinstance(raw, str) or not raw:
        return ""

    # full SVG
    m = re.search(r'(<svg[\s\S]*?</svg>)', raw, re.IGNORECASE)
    if m:
        return m.group(1).strip()

    # partial SVG starting with <svg but no closing tag yet
    m2 = re.search(r'(<svg[\s\S]+)', raw, re.IGNORECASE)
    if m2:
        return m2.group(1).strip()

    return ""

def safe_xml_repair(svg: str) -> str:
    if not isinstance(svg, str):
        return FALLBACK_SVG

    start = svg.find("<svg")
    if start == -1:
        return FALLBACK_SVG
    svg = svg[start:]

    # remove illegal control chars
    svg = "".join(ch for ch in svg if ord(ch) in (9, 10, 13) or ord(ch) >= 32)

    # compact whitespace between tags
    svg = re.sub(r">\s+<", "><", svg)
    svg = re.sub(r"\s+", " ", svg).strip()

    # add xmlns if missing
    if "xmlns=" not in svg:
        svg = svg.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)

    # add viewBox if missing
    if "viewBox=" not in svg:
        svg = re.sub(
            r"<svg([^>]*)>",
            r'<svg\1 viewBox="0 0 256 256">',
            svg,
            count=1
        )

    # if already has a full svg block, keep only that
    m = re.search(r"(<svg[\s\S]*?</svg>)", svg, re.IGNORECASE)
    if m:
        svg = m.group(1)
    else:
        # otherwise close at last complete tag boundary
        last_gt = svg.rfind(">")
        if last_gt == -1:
            return FALLBACK_SVG
        svg = svg[: last_gt + 1] + "</svg>"

    # escape stray &
    svg = re.sub(
        r'&(?!amp;|lt;|gt;|quot;|apos;|#\d+;|#x[0-9A-Fa-f]+;)',
        '&amp;',
        svg
    )

    # enforce path limit early
    if svg.count("<path") > 256:
        return FALLBACK_SVG

    # enforce length safely
    if len(svg) > 8000:
        prefix = svg[:7990]
        cut = prefix.rfind(">")
        if cut == -1:
            return FALLBACK_SVG
        svg = prefix[: cut + 1]
        if not svg.lower().endswith("</svg>"):
            svg += "</svg>"

    # iterative XML fix: trim to previous complete tag if parse fails
    for _ in range(8):
        try:
            ET.fromstring(svg)
            if len(svg) > 8000 or svg.count("<path") > 256:
                return FALLBACK_SVG
            return svg
        except Exception:
            close_pos = svg.lower().rfind("</svg>")
            if close_pos == -1:
                return FALLBACK_SVG
            prefix = svg[:close_pos]
            last_gt = prefix.rfind(">")
            if last_gt == -1:
                return FALLBACK_SVG
            svg = prefix[: last_gt + 1] + "</svg>"

    return FALLBACK_SVG

model.config.use_cache = True
model.eval()

@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=700):
    formatted = [build_prompt_text(p) for p in prompts]

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=192,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,   # greedy = faster + more stable
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    # with LEFT padding, slice after the padded prompt length
    prompt_len = inputs["input_ids"].shape[1]

    batch_svgs = []
    raw_texts = []

    for output in outputs:
        gen_tokens = output[prompt_len:]
        raw = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        raw_texts.append(raw)

        svg = extract_svg(raw)
        svg = safe_xml_repair(svg)

        batch_svgs.append(svg)

    return batch_svgs, raw_texts

preds = []
raw_generations = []

for start in range(0, len(test_df), INFERENCE_BATCH):
    batch_prompts = test_df["prompt"].iloc[start:start + INFERENCE_BATCH].tolist()
    batch_svgs, batch_raws = generate_batch(batch_prompts, max_new_tokens=700)
    preds.extend(batch_svgs)
    raw_generations.extend(batch_raws)

print("Inference complete.")
print("Rows:", len(preds))
print("Unique SVGs:", len(set(preds)))
print("Fallback count:", sum(x == FALLBACK_SVG for x in preds))
print("Median SVG length:", int(np.median([len(x) for x in preds])))
print("Max SVG length:", max(len(x) for x in preds))

In [ ]:
# ============================================================
# CELL 12 - RETRY FALLBACK ROWS (two-pass sampling)
# ============================================================
# Runs automatically after main inference (Cell 11).
# Pass 1: temperature=0.3 — conservative sampling
# Pass 2: temperature=0.6 — more creative, for rows still failing
# Only fallback rows are retried; all other preds are untouched.

@torch.inference_mode()
def retry_generate(prompts, max_new_tokens=900, temperature=0.3):
    formatted = [build_prompt_text(p) for p in prompts]
    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=192,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    prompt_len = inputs["input_ids"].shape[1]
    results = []
    for output in outputs:
        gen_tokens = output[prompt_len:]
        raw = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        svg = extract_svg(raw)
        svg = safe_xml_repair(svg)
        results.append(svg)
    return results


RETRY_BATCH = max(1, INFERENCE_BATCH // 2)   # smaller batch for safety

for pass_num, temp in enumerate([0.3, 0.6], start=1):
    fallback_idx = [i for i, x in enumerate(preds) if x == FALLBACK_SVG]
    if not fallback_idx:
        print(f"Pass {pass_num}: no fallbacks remaining — done early.")
        break

    print(f"\nPass {pass_num} (temp={temp}): retrying {len(fallback_idx)} fallback rows ...")
    recovered = 0

    for start in range(0, len(fallback_idx), RETRY_BATCH):
        batch_idx     = fallback_idx[start : start + RETRY_BATCH]
        batch_prompts = [test_df.iloc[i]["prompt"] for i in batch_idx]
        retried       = retry_generate(batch_prompts, temperature=temp)

        for idx, new_svg in zip(batch_idx, retried):
            if new_svg != FALLBACK_SVG:
                preds[idx] = new_svg
                raw_generations[idx] = "(retry pass {})".format(pass_num)
                recovered += 1

    print(f"  Recovered: {recovered} / {len(fallback_idx)}")

remaining_fb = sum(x == FALLBACK_SVG for x in preds)
print(f"\nFallbacks remaining after all retry passes: {remaining_fb} / {len(preds)}")
print(f"Unique SVGs now: {len(set(preds))}")


In [ ]:
# ============================================================
# CELL 13 - POST-RETRY STATS
# ============================================================

import numpy as np

fallback_count = sum(x == FALLBACK_SVG for x in preds)
unique_svgs    = len(set(preds))
lengths        = [len(x) for x in preds]
path_counts    = [x.count("<path") for x in preds]

print("=== Post-retry summary ===")
print(f"  Total rows      : {len(preds)}")
print(f"  Fallbacks left  : {fallback_count}  ({100*fallback_count/len(preds):.1f}%)")
print(f"  Unique SVGs     : {unique_svgs}")
print(f"  Median length   : {int(np.median(lengths))}")
print(f"  Max length      : {max(lengths)}")
print(f"  Over 8000 chars : {sum(l > 8000 for l in lengths)}  (must be 0)")
print(f"  Over 256 paths  : {sum(p > 256 for p in path_counts)}  (must be 0)")


In [ ]:
# ============================================================
# CELL 14 - FINAL SAFETY CHECK + SAVE SUBMISSION
# ============================================================

import xml.etree.ElementTree as ET

def final_xml_check(svg: str):
    try:
        ET.fromstring(svg)
        return True
    except Exception:
        return False

# One final repair pass before saving
preds = [safe_xml_repair(x) for x in preds]

valid_xml = sum(final_xml_check(x) for x in preds)
fallback_count = sum(x == FALLBACK_SVG for x in preds)
unique_svgs = len(set(preds))
lengths = [len(x) for x in preds]

print("After final repair pass:")
print("Rows:", len(preds))
print("Valid XML:", valid_xml)
print("Fallback count:", fallback_count)
print("Unique SVGs:", unique_svgs)
print("Median SVG length:", int(np.median(lengths)))
print("Max SVG length:", max(lengths))
print("Any >8000 chars:", any(len(x) > 8000 for x in preds))
print("Any >256 paths:", any(x.count("<path") > 256 for x in preds))

submission = pd.DataFrame({
    "id": test_df["id"],
    "svg": preds
})

submission.to_csv(SUBMISSION_PATH, index=False)

print("\nSaved:", SUBMISSION_PATH)
display(submission.head(3))

In [ ]:
# from google.colab import files
# files.download('submission.csv')